In [1]:
import requests
import torch.nn as nn
import torch
from transformers import GPT2TokenizerFast


url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
text = response.text

print(f"Dataset length: {len(text):,} characters")



Dataset length: 1,115,394 characters


In [2]:
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
encoded_text = tokenizer.encode(text)

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


In [3]:
from torch.utils.data import Dataset, DataLoader

class Dataset:
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.output_ids = []

        txt_ids = tokenizer.encode(txt)

        for i in range(0, len(txt_ids) - max_length, stride):
            input_chunk = txt_ids[i:i+max_length]
            target_chunk = txt_ids[i+1 : i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.long))
            self.output_ids.append(torch.tensor(target_chunk, dtype=torch.long))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.output_ids[idx]

def create_dataloader(txt, batch_size=16, max_length=16, stride=8, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
    dataset = Dataset(txt, tokenizer, max_length, stride)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)


In [4]:
dataloader = create_dataloader(text)
first_batch = next(iter(dataloader))

Token indices sequence length is longer than the specified maximum sequence length for this model (338025 > 1024). Running this sequence through the model will result in indexing errors


In [5]:
first_batch[0].type()  # torch.LongTensor

'torch.LongTensor'

In [ ]:
class LSTM_LM(nn.Module):
    def __init__(self, vocab_size,emb_size,  hidden_size):
        super(LSTM_LM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.W_x = nn.Linear(emb_size, hidden_size)
        self.W_f = nn.Linear(2 * hidden_size, hidden_size)
        self.W_i = nn.Linear(2 * hidden_size, hidden_size)
        self.W_c = nn.Linear(2 * hidden_size, hidden_size)
        self.W_o = nn.Linear(2 * hidden_size, hidden_size)
        self.vocab_projection = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h_t_minus_one, C_t_minus_one):
        
        _, seq_length = x.shape

        outputs = []
        h_t = h_t_minus_one
        C_t = C_t_minus_one

        for t in range(seq_length):
            x_t = self.embedding(x[:, t])
            x_t = self.W_x(x_t)
            f_t = torch.sigmoid(self.W_f(torch.cat((h_t, x_t), dim=1)))
            i_t = torch.sigmoid(self.W_i(torch.cat((h_t, x_t), dim=1)))
            C_tilda_t = torch.tanh(self.W_c(torch.cat((h_t, x_t), dim=1)))
            
            C_t = f_t * C_t + i_t * C_tilda_t
            o_t = torch.sigmoid(self.W_o(torch.cat((h_t, x_t), dim=1)))
            h_t = o_t * torch.tanh(C_t)

            outputs.append(C_t.unsqueeze(1))
        
        outputs = torch.cat(outputs, dim=1)
        logits = self.vocab_projection(outputs)

        return logits, (C_t, h_t)



In [9]:
vocab_size = tokenizer.vocab_size
emb_dim = 64
hidden_dim = 128
model = LSTM_LM(vocab_size, emb_dim, hidden_dim)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 10

In [10]:
model.train()
print(f"Training on {device}...")
for epoch in range(num_epochs):
    total_loss = 0
    for input_ids, target_ids in dataloader:
        input_ids, target_ids = input_ids.to(device), target_ids.to(device)
        batch_size = input_ids.size(0)
        h_0 = torch.zeros(batch_size, hidden_dim).to(device)
        C_0 = torch.zeros(batch_size, hidden_dim).to(device)

        outputs, _ = model(input_ids, h_0, C_0)
        # reshape for cross entropy: (batch*seq, vocab)
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            target_ids.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

Training on cuda...
Epoch 1, Loss: 5.0502
Epoch 2, Loss: 4.1650
Epoch 3, Loss: 3.8316
Epoch 4, Loss: 3.6069
Epoch 5, Loss: 3.4399
Epoch 6, Loss: 3.3066
Epoch 7, Loss: 3.1963
Epoch 8, Loss: 3.1034
Epoch 9, Loss: 3.0239
Epoch 10, Loss: 2.9536


In [14]:
def generate(model, tokenizer, start_text, max_new_tokens=500, device='cuda'):
    model.eval()
    device = torch.device(device if torch.cuda.is_available() else 'cpu')

    input_ids = torch.tensor(tokenizer.encode(start_text), dtype=torch.long).unsqueeze(0).to(device)
    h_t = torch.zeros(1, 128).to(device)
    C_t = torch.zeros(1, 128).to(device)

    generated = input_ids.tolist()[0]

    for _ in range(max_new_tokens):
        logits_seq, (C_t, h_t) = model(input_ids, h_t, C_t)
        next_token_logits = logits_seq[0, -1, :]  # last time step
        # Apply temperature (optional, controls randomness)
        temperature = 1.0
        probs = torch.softmax(next_token_logits / temperature, dim=-1)
        
        # Sample a token from the probability distribution
        next_token = torch.multinomial(probs, num_samples=1).item()

        generated.append(next_token)
        input_ids = torch.tensor([[next_token]], dtype=torch.long).to(device)  # keep on same device

    return tokenizer.decode(generated)



generated_text = generate(model, tokenizer,"Today")

print(generated_text)

torch.save(model.state_dict(), "./models/lstm_lm.pth")

Today are the dire tempted of his grow, you mark
Corioli' sake, and there lies the end
of empty attempt in the world,
Whichaying, after more care the better,--
My take-bellied men hear this hour
The other bet our task.

SLY:
And I, with thy weapon drop with a Jack-path eye.'

ANGELO:
Tybalt, sir: let us send the world over. How came?

KATHARINA:
Well, then.

DUKE OF AUMERLE:
That's do now you at all.

DUKE OF YORK:
Thy father and jealousies in pride;
For shame, I dare not know. You are many?

CAMILLO:
Be hostages toad,'
And this distilled liquor dozen thin of seven sons,
Lest short shrift to his conditions and for his
bed pity Claudio's corse, unless you fast me justice;
I am imprison'd in manhood stole brought me some festival
And patience behold to that ignoble plants.
It shall be put down,
To say you, and take leave my hair in my sorrow's grave
Stays, you with lay good gods: I cannot
I saw myself aminate; who were the been done a jade
I know thy brethren roar'd: this is impossible
M